RMRB Analysis v2
- (2024.9.12) Even with plaintext and ciphertext extraction, many ads remain 'invisible' because some ads do not show the text format "广告" on pdf. Morever, while we can use OCR tech to recognize the non-textual "广告" though, it may lead to lower efficiency and longer time. To solve this problem, in version 2, we introduce a new tech named "Computer vision", which can detect ad block on a image.
- As we all know that most RMRB's half-page ads occupy about half area of a single layout. According to this fact, we can use CV to detect blocks which occupy a specific( ≈ image area / 2).
- (2024.9.13) Add PDF to Image conversion function `PDF2Image`
- Add `CV_Detect_Ads`

In [1]:
from PyPDF2 import PdfReader, PdfWriter
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from datetime import datetime, timedelta
from pathlib import Path
import os
import cv2
import numpy as np

from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.converter import PDFPageAggregator
from pdfminer.layout import LTTextBoxHorizontal, LAParams
from pdfminer.high_level import extract_text
# from pdfminer.pdfinterp import PDFTextExtractionNotAllowed

# Set the path to the Tesseract installation
pytesseract.pytesseract.tesseract_cmd = "D:\\Tesseract-OCR\\tesseract.exe"

In [2]:
Advertisement = "广告"
Cipher_AD = """!!"!"""

In [3]:
ROOT_PATH = "D:/AI_data_analysis/RMRB/"
RMRB2024090616 = "D:/AI_data_analysis/RMRB/2024090616.pdf"
def PATH_FUN(YEAR, MONTH, DAY, EDITION, SUFFIX, NAME_bool=False):
    Split_Year = ["2023"] 
    NAME = YEAR + MONTH + DAY
    INFO_PATH = (
        NAME 
        + (
            ("/" + NAME + EDITION) 
            if YEAR in Split_Year else ""
        ))
    SUFFIX = ".pdf"
    pdf_path = ROOT_PATH + YEAR + "/" + INFO_PATH + SUFFIX
    if NAME_bool:
        return pdf_path, NAME
    else: return  pdf_path
PDF_PATH = PATH_FUN("2022", "01", "02", "02", ".pdf")
PDF_PATH

'D:/AI_data_analysis/RMRB/2022/20220102.pdf'

In [4]:
def Generate_Dates(year):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year + 1, 1, 1)
    current_date = start_date
    
    dates = []
    while current_date < end_date:
        dates.append(current_date.strftime('%Y%m%d'))
        current_date += timedelta(days=1)
    return dates

def Check_PDF_Exist(YEAR):
    MISSING_PDF = []
    start_date = datetime(int(YEAR), 1, 1)
    end_date = datetime(int(YEAR) + 1, 1, 1)
    current_date = start_date
    while current_date < end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PATH = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf")
        try:
            with open(PATH, "rb") as file:
                PdfReader(file)
        except FileNotFoundError: 
            print(PATH)
            MISSING_PDF.append(PATH)
        current_date += timedelta(days=1)
        return MISSING_PDF
# Check_PDF_Exist("2017")

In [5]:
def Count_Files(folder_path):
    # Create a Path object for the folder
    folder = Path(folder_path)
    # Count all files in the folder
    file_count = sum(1 for file in folder.iterdir() if file.is_file())
    return file_count
# Count_Files("D:/AI_data_analysis/RMRB")

def Check_Folder_Exist(folder_path):
    # Check if the folder exists
    if not os.path.exists(folder_path):
        # Create the folder if it does not exist
        os.makedirs(folder_path)
        print(f'Folder created: {folder_path}')
    else:
        print(f'Folder already exists: {folder_path}')
# Check_Folder_Exist("D:/AI_data_analysis/RMRB")

In [6]:
def PDF2Image(pdf_path: str, actual_page_list):
    PDF = PdfReader(pdf_path)
    root_path = "/".join(pdf_path.split("/")[:-1]) + "/"
    Name = pdf_path.split("/")[-1].split(".")[0]
    num_of_pages = len(PDF.pages)
    print(f"PDF page(s) num: {num_of_pages}" + "\n")
    print("PDF PATH:", PDF_PATH)
    # Convert PDF pages to images
    page_list = [num - 1 for num in actual_page_list]

    for page_num in page_list:
        IMAGE = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
        page_num_str = ("0" + str(page_num + 1)) if len(str(page_num + 1)) == 1 else str(page_num + 1)
        image_path = root_path + Name + f"{page_num_str}.png"
        IMAGE[0].save(image_path, "PNG")
        print(f"Page {page_num + 1} image saved to: {image_path}")
# PDF2Image("D:/AI_data_analysis/RMRB/20200117.pdf", [5, 12, 18, 20])

In [7]:
# IMAGE_TEST = convert_from_path("D:/AI_data_analysis/RMRB/20200117.pdf")

In [8]:
def CV_Detect_Ads_Test(
        root_path: str, 
        pdf_name: str, 
        pdf_version: str, 
        image_element, 
        Threshold: list=[0.4, 0.6], 
        Image_Clip_Bool: bool=False
    ):
    pdf_path = root_path + pdf_name + ".pdf"

    # Convert Pillow image to numpy array
    image_np = np.array(image_element)
    # Convert RGB to BGR since OpenCV uses BGR format
    image = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    # Get the dimensions of the image
    height, width = image_np.shape[:2]
    # Calculate the area of the entire image
    image_area = height * width
    print(f"Version {pdf_version} Image size (x, y) = ({width}, {height})")
    print(f"Version {pdf_version} Image area:", image_area)
    Ad_Block_Threshold_Min = Threshold[0] * image_area
    Ad_Block_Threshold_Max = Threshold[1] * image_area
    print(f"Ad block area threshold: Min = {Ad_Block_Threshold_Min}, Max = {Ad_Block_Threshold_Max}")

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Apply edge detection
    edges = cv2.Canny(blurred, 50, 150)

    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ad_found = False  # Flag to indicate if an ad is detected

    # Iterate through contours
    for contour in contours:
        # print(contour)
        # Get the bounding box of the contour
        x, y, w, h = cv2.boundingRect(contour)
        # Calculate the area of the bounding rectangle
        area_rect = w * h

        # Filter based on area and aspect ratio (adjust thresholds as needed)
        if area_rect >= Ad_Block_Threshold_Min and area_rect <= Ad_Block_Threshold_Max:
            ad_found = True
            # Draw a rectangle around the detected ad block
            cv2.rectangle(image, (x - 10, y - 10), (x + w + 10, y + h + 10), (0, 255, 0), 2)
            
            if Image_Clip_Bool: 
                # Save the ad block image to local
                ad_block = image[y:y + h, x:x + w] # Clip the ad block
                output_block_path = root_path + pdf_name + pdf_version + "_AD_Block.png"
                cv2.imwrite(output_block_path, ad_block)
                print(f"Version {pdf_version} Saved ad block image output to {output_block_path}")
            print(f"Version {pdf_version} Detected Ad Block: Bounding Box Area = {area_rect}")
            print(f"Version {pdf_version} Ad Block position: (x, y, w, h) = ({x}, {y}, {w}, {h})")

    # Display the result or save it for later review
    if ad_found:
        output_path = root_path + pdf_name + pdf_version + "_CV.png"
        cv2.imwrite(output_path, image)  # Save image with detected ads
        print(f"Version {pdf_version} Advertisements detected in {pdf_path}")
        print(f"Version {pdf_version} Saved whole image output to {output_path}")
    else:
        print(f"No advertisements detected in {pdf_path}")
# CV_Detect_Ads_Test("D:/AI_data_analysis/RMRB/", "20200117", "12", IMAGE_TEST[11])

In [9]:
def CV_Detect_Ads(
        root_path: str,
        pdf_path: str, 
        pdf_version: str, 
        image_element, 
        Threshold: list=[0.4, 0.6], 
        Image_Clip_Bool: bool=False
    ):
    """
    pdf_path: pdf path

    pdf_version: version of a pdf for image conversion

    image_element: the image element which generated from `convert_from_path()`

    Threshold list: [min, max] (area thershold is [min * whole_area, max * whole_area])

    Image_Clip_Bool: Whether clip the detected image to an individual image. Default False.
    """
    pdf_name = pdf_path.split("/")[-1].split(".")[0]

    # Convert Pillow image to numpy array
    image_np = np.array(image_element)
    # Convert RGB to BGR since OpenCV uses BGR format
    image = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    # Get the dimensions of the image
    height, width = image_np.shape[:2]
    # Calculate the area of the entire image
    image_area = height * width
    Ad_Block_Threshold_Min = Threshold[0] * image_area
    Ad_Block_Threshold_Max = Threshold[1] * image_area

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Apply edge detection
    edges = cv2.Canny(blurred, 50, 150)

    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ad_found = False  # Flag to indicate if an ad is detected

    # Iterate through contours
    for contour in contours:
        # print(contour)
        # Get the bounding box of the contour
        x, y, w, h = cv2.boundingRect(contour)
        # Calculate the area of the bounding rectangle
        area_rect = w * h

        # Filter based on area and aspect ratio (adjust thresholds as needed)
        if area_rect >= Ad_Block_Threshold_Min and area_rect <= Ad_Block_Threshold_Max:
            ad_found = True
            # Draw a rectangle around the detected ad block
            cv2.rectangle(image, (x - 10, y - 10), (x + w + 10, y + h + 10), (0, 255, 0), 2)
            
            if Image_Clip_Bool: 
                # Save the ad block image to local
                ad_block = image[y:y + h, x:x + w] # Clip the ad block
                output_block_path = root_path + pdf_name + "_" + pdf_version + "_AD_Block.png"
                cv2.imwrite(output_block_path, ad_block)
                print(f"Version {pdf_version} Saved ad block image to {output_block_path}")

    # Display the result or save it for later review
    if ad_found:
        output_path = root_path + pdf_name + "_" + pdf_version + "_CV.png"
        cv2.imwrite(output_path, image)  # Save image with detected ads
        print(f"Version {pdf_version} Saved whole image to {output_path}")
# CV_Detect_Ads("D:/AI_data_analysis/RMRB/", "D:/AI_data_analysis/RMRB/20200117.pdf", "12", IMAGE_TEST[11])

In [10]:
# Generate image for all ad pages using OCR
def Genetare_AD_Image(YEAR, Begin_date="0101", End_date="1231"):
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD/"
    Check_Folder_Exist(FOLD_PATH)
    start_date = datetime(int(YEAR), int(Begin_date[:2]), int(Begin_date[2:]))
    end_date = datetime(int(YEAR), int(End_date[:2]), int(End_date[2:]))
    print(f"Begin date: {YEAR + Begin_date}, End date: {YEAR + End_date}")
    current_date = start_date
    while current_date <= end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PDF_PATH, NAME = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf", NAME_bool=True)
        print("PDF PATH:", PDF_PATH)
        with pdfplumber.open(PDF_PATH) as PDF:
            # PDF = PdfReader(file)
            PAGES_NUM = len(PDF.pages)
            IMAGE = convert_from_path(PDF_PATH)
            for page_num in range(PAGES_NUM):
                text = PDF.pages[page_num].extract_text().replace(" ", "")
                IMAGE_PATH = FOLD_PATH + f"{NAME}_{page_num+1}_AD.png"
                if Advertisement in text: # First filter: plaintext "广告"
                    # Attension: first_page and last_page starts at 1
                    IMAGE[page_num].save(IMAGE_PATH, "PNG") # Only one image in `IMAGE`
                    print("Full-page Ad:", IMAGE_PATH)
                elif """!!"!""" in text: # Second filter: ciphertext "广告"
                    # Attension: first_page and last_page starts at 1
                    IMAGE[page_num].save(IMAGE_PATH, "PNG") # Only one image in `IMAGE`
                    print("Half-page Ad", IMAGE_PATH)
                else:
                    CV_Detect_Ads(FOLD_PATH, PDF_PATH, str(page_num+1), IMAGE[page_num])
        current_date += timedelta(days=1)
Genetare_AD_Image("2017", "0824")
# 2018: 6 months: 158 mins

Folder already exists: D:/AI_data_analysis/RMRB/2017_AD/
Begin date: 20170824, End date: 20171231
PDF PATH: D:/AI_data_analysis/RMRB/2017/20170824.pdf
Version 11 Saved whole image to D:/AI_data_analysis/RMRB/2017_AD/20170824_11_CV.png
Version 12 Saved whole image to D:/AI_data_analysis/RMRB/2017_AD/20170824_12_CV.png
Version 14 Saved whole image to D:/AI_data_analysis/RMRB/2017_AD/20170824_14_CV.png
Version 15 Saved whole image to D:/AI_data_analysis/RMRB/2017_AD/20170824_15_CV.png
Version 17 Saved whole image to D:/AI_data_analysis/RMRB/2017_AD/20170824_17_CV.png
Full-page Ad: D:/AI_data_analysis/RMRB/2017_AD/20170824_19_AD.png
Full-page Ad: D:/AI_data_analysis/RMRB/2017_AD/20170824_24_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2017/20170825.pdf
Version 5 Saved whole image to D:/AI_data_analysis/RMRB/2017_AD/20170825_5_CV.png
Full-page Ad: D:/AI_data_analysis/RMRB/2017_AD/20170825_6_AD.png
Version 10 Saved whole image to D:/AI_data_analysis/RMRB/2017_AD/20170825_10_CV.png
Version 11 Sa